# 05 — Análise de Sensibilidade, Teste de Estresse e Gatilhos de Governança

**Regra de Governança:**
* O **Cenário Base (`-R$ 10,00M` líquidos)** é a **única alocação aprovada**.
* O **Cenário Adverso (`-20%` volume / `+15%` custo, levando a `-R$ 12,24M` líquidos)** é um **teste de estresse** que aciona gatilhos automáticos:
  1. Pausar ou reduzir incentivos nas cidades com pior retorno marginal;
  2. Priorizar `City 2` e `City 3` (maior MC incremental por R$ 1 de custo bruto);
  3. Reduzir ou pausar capacidade em `City 17` se o SLA não melhorar;
  4. Revisar a alocação mensalmente com dados reais.


In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

PROJECT_DIR = "/content/drive/MyDrive/ifood-growth-competition-case"
if os.path.exists("/content/drive"):
    if not os.path.exists(PROJECT_DIR):
        from google.colab import drive
        drive.mount("/content/drive")
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        PROJECT_DIR = os.getcwd()

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)

for sub in ["data/raw", "data/processed", "outputs/excel", "outputs/figures", "outputs/reports", "dashboard"]:
    os.makedirs(sub, exist_ok=True)

print("Diretório ativo:", os.getcwd())


In [ ]:
df_sens = pd.DataFrame([
    {
        "cenario": "Adverso (Teste de Estresse: -20% vol / +15% custo)",
        "status": "Teste de estresse (nao aprovado)",
        "investimento_bruto_rs": round(11115969.51 * 1.15, 2),
        "pedidos_incrementais_liquidos": round(352703.38 * 0.80, 2),
        "gmv_incremental_rs": 20863675.02,
        "mc_incremental_rs": 1680164.19,
        "custo_operacional_sla_rs": round(984235.73 * 1.15, 2),
        "impacto_liquido_om_rs": -12244971.84,
        "impacto_liquido_om_m": -12.24,
        "acao_guardrail": "Acionar guardrails: pausar/reduzir incentivos de pior retorno marginal, priorizar City 2 e City 3, reduzir/pausar capacidade em City 17 se SLA nao melhorar, revisar mensalmente."
    },
    {
        "cenario": "Base (Alocacao Aprovada)",
        "status": "Alocacao oficial aprovada",
        "investimento_bruto_rs": 11115969.51,
        "pedidos_incrementais_liquidos": 352703.38,
        "gmv_incremental_rs": 26079593.78,
        "mc_incremental_rs": 2100205.24,
        "custo_operacional_sla_rs": 984235.73,
        "impacto_liquido_om_rs": -10000000.00,
        "impacto_liquido_om_m": -10.00,
        "acao_guardrail": "Cumpre exatamente a restricao de -R$ 10,00M na Operating Margin e todos os guardrails."
    },
    {
        "cenario": "Otimista (+20% vol / -10% custo)",
        "status": "Oportunidade de upside",
        "investimento_bruto_rs": round(11115969.51 * 0.90, 2),
        "pedidos_incrementais_liquidos": round(352703.38 * 1.20, 2),
        "gmv_incremental_rs": 31295512.54,
        "mc_incremental_rs": 2520246.29,
        "custo_operacional_sla_rs": round(984235.73 * 0.90, 2),
        "impacto_liquido_om_rs": -8369938.43,
        "impacto_liquido_om_m": -8.37,
        "acao_guardrail": "Gera folga de R$ 1,63M na Operating Margin; monitorar teto de 98% em City 6."
    }
])
df_sens.to_csv("outputs/reports/sensibilidade_cenarios.csv", index=False)
display(df_sens)

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(df_sens["cenario"], df_sens["impacto_liquido_om_m"], color=["#DC2626", "#0F172A", "#15803D"], width=0.45)
ax.axhline(-10.0, color="#EA1D2C", linestyle="--", linewidth=1.2, label="Restrição do Case (-R$ 10,00M)")
ax.set_title("Impacto Líquido na Operating Margin por Cenário (R$ Milhões)", fontweight="bold")
ax.set_ylabel("Impacto Líquido OM (R$ Milhões)")
ax.set_ylim(-14.5, 0)
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() - 0.85, f"R$ {b.get_height():.2f}M", ha="center", fontweight="bold", color="#111827")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("outputs/figures/05_sensibilidade_om.png", dpi=180)
plt.show()
